<a href="https://colab.research.google.com/github/jiwonleelee/Deepfake-Detection-v2/blob/main/notebooks/01_preprocessing/CDF_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install insightface onnxruntime-gpu

In [6]:
import cv2
import shutil
import numpy as np
import os
import zipfile
import tempfile
import random
import gc
from pathlib import Path
from tqdm import tqdm
from insightface.app import FaceAnalysis
from google.colab import drive
from collections import defaultdict

In [3]:
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
import sys
# ff_utils.py가 저장된 '폴더' 경로를 시스템 경로에 추가
MODULE_PATH = "/content/drive/MyDrive/Deepfake-Detection-v2/src"
if MODULE_PATH not in sys.path:
    sys.path.append(MODULE_PATH)

# 이제 임포트가 가능합니다
import pp_utils
import processor
import importlib
importlib.reload(pp_utils) # 코드를 수정하고 다시 저장했을 때 반영하기 위함
importlib.reload(processor)

<module 'processor' from '/content/drive/MyDrive/Deepfake-Detection-v2/src/processor.py'>

In [8]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [9]:
ZIP_PATH = "/content/drive/MyDrive/Deepfake-Detection-v2/Datasets/CDF/Celeb-DF-v2.zip"
GDRIVE_SAVE_BASE = "/content/drive/MyDrive/Deepfake-Detection-v2/Datasets/CDF"
FINAL_SAVE_PATH = os.path.join(GDRIVE_SAVE_BASE, "Images")

NUM_FRAMES = 15
TARGET_SIZE = (224, 224)
SAMPLE_SIZE = 100 # Real 100개 뽑으면 Fake 100개가 자동으로 매칭됨

In [10]:
# providers=['CUDAExecutionProvider']로 GPU 강제 사용
detector = FaceAnalysis(allowed_modules=['detection', 'landmark_2d'], providers=['CUDAExecutionProvider'])
detector.prepare(ctx_id=0, det_size=(640, 640))

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 96707.54KB/s] 


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvid

In [12]:
def run_preprocessing():
    # 1. 저장 경로 초기화
    if os.path.exists(FINAL_SAVE_PATH):
        shutil.rmtree(FINAL_SAVE_PATH)
    os.makedirs(FINAL_SAVE_PATH, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        # 2. 데이터 리스트 확보 및 정렬 (재현성 확보)
        all_videos = [
            f for f in z.namelist()
            if f.lower().endswith('.mp4') and not os.path.basename(f).startswith('._')
        ]
        all_videos.sort()

        # 3. 데이터 필터링 및 1:1 타겟 매칭
        real_videos_all = [f for f in all_videos if 'celeb-real' in f.lower()]
        fake_all = [f for f in all_videos if 'celeb-synthesis' in f.lower()]

        fake_by_target = defaultdict(list)
        for fake_path in fake_all:
            fake_stem = Path(fake_path).stem # 예: id10_id0_0001
            parts = fake_stem.split('_')

            if len(parts) >= 3:
                # 첫 번째 ID(원본)와 마지막 번호(비디오)를 결합
                # id10 + _ + 0001 = id10_0001
                true_target_id = f"{parts[0]}_{parts[-1]}"
                fake_by_target[true_target_id].append(fake_path)

        matched_pairs = []
        for real_path in real_videos_all:
            target_id = Path(real_path).stem
            candidates = fake_by_target.get(target_id, [])
            if candidates:
                # 시드 고정으로 인한 결정론적 선택
                chosen_fake = random.choice(candidates)
                matched_pairs.append((real_path, chosen_fake))

        # 4. 크로스 검증용 샘플링 (100쌍 추출)
        num_to_sample = min(len(matched_pairs), SAMPLE_SIZE)
        if num_to_sample > 0:
            selected_pairs = random.sample(matched_pairs, num_to_sample)
        else:
            print("❌ 매칭된 영상 쌍이 없습니다. 구조를 확인하세요.")
            return

        real_videos = [p[0] for p in selected_pairs]
        fake_videos = [p[1] for p in selected_pairs]

        print(f"📊 전처리 시작 - Real: {len(real_videos)}개, Fake: {len(fake_videos)}개 (Target 매칭 완료)")

        # 5. 범용 프로세서(src/processor.py) 호출
        # 크로스 검증용이므로 TARGET_SIZE=(224, 224), margin_rate=0.125 적용

        # [Real 데이터 처리]
        processor.process_video_list(
            video_list=real_videos,
            save_base_path=FINAL_SAVE_PATH,
            label_type="Real",
            detector=detector,
            pp_utils=pp_utils,
            num_frames=NUM_FRAMES,
            target_size=(224, 224), # 검증용 사이즈 224로 변경
            margin_rate=0.125,      # 타이트한 마진 적용
            zip_obj=z
        )

        # [Fake 데이터 처리]
        processor.process_video_list(
            video_list=fake_videos,
            save_base_path=FINAL_SAVE_PATH,
            label_type="Fake",
            detector=detector,
            pp_utils=pp_utils,
            num_frames=NUM_FRAMES,
            target_size=(224, 224), # 검증용 사이즈 224로 변경
            margin_rate=0.125,      # 타이트한 마진 적용
            zip_obj=z
        )

run_preprocessing()

📊 전처리 시작 - Real: 100개, Fake: 100개 (Target 매칭 완료)


Processing Fake: 100%|██████████| 100/100 [01:50<00:00,  1.11s/it]


In [13]:
SRC_DIR = "/content/drive/MyDrive/Deepfake-Detection-v2/Datasets/CDF/Images"
DST_DIR = "/content/drive/MyDrive/Deepfake-Detection-v2/Datasets/CDF"
ZIP_NAME = "CDF.zip"

os.makedirs(DST_DIR, exist_ok=True)

zip_path = os.path.join(DST_DIR, ZIP_NAME)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(SRC_DIR):
        for file in files:
            file_path = os.path.join(root, file)

            # zip 내부 경로 (Celeb_frames/... 구조 유지)
            arcname = os.path.relpath(file_path, SRC_DIR)

            zipf.write(file_path, arcname)

print(f"✅ 압축 완료: {zip_path}")


✅ 압축 완료: /content/drive/MyDrive/Deepfake-Detection-v2/Datasets/CDF/CDF.zip
